In [1]:
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from vectorization.vectorize import SBERTVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
import pickle
import os


C:\Users\fidel\miniconda3\envs\ML-XAI\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ag_data = load_dataset("ag_news")

X_train = ag_data['train']["text"]
y_train = ag_data['train']['label']

In [3]:
vectorizers = {
        'tfidf': TfidfVectorizer(
            max_features=10000,
            ngram_range=(1,2),
            stop_words='english',
            min_df=2,
            max_df=0.95
        ),
        'sbert': SBERTVectorizer(model_name='all-MiniLM-L6-v2')
    }

In [4]:
models = {
    'svm': SVC(kernel='linear', random_state=42, probability=True),
    'mlp': MLPClassifier(
        hidden_layer_sizes=(100, 50),
        max_iter=500,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1
    ),
    'dt': RandomForestClassifier(
        n_estimators=200,
        max_depth=30,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        class_weight='balanced',
        n_jobs=-1,
        random_state=42
    )
}

In [ ]:
output_dir = "saved_pipelines_AGnews"
for vect_name, vect in vectorizers.items():
    for model_name, model in models.items():
        print(f"Training: {vect_name} + {model_name}")

        pipeline = Pipeline([
            ('vectorizer', vect),
            ('classifier', model)
        ])

        try:
            pipeline.fit(X_train, y_train)

            filename = f"{vect_name}_{model_name}.pkl"
            filepath = os.path.join(output_dir, filename)

            with open(filepath, 'wb') as f:
                pickle.dump(pipeline, f)

            print(f"Saved: {filename}")
        except Exception as e:
            print(f"Failed: {vect_name} + {model_name} — {e}")

Training: tfidf + svm
Saved: tfidf_svm.pkl
Training: tfidf + mlp
Saved: tfidf_mlp.pkl
Training: tfidf + dt
Saved: tfidf_dt.pkl
Training: sbert + svm


Batches: 100%|██████████| 3750/3750 [17:35<00:00,  3.55it/s]
